# SkylineGeolocation — Avenue A/B/C GPU Evaluation

Tests three avenues that require GPU:
- **Avenue A**: SAM 2 zero-shot sky segmentation (full-res 1080×720 masks)
- **Avenue B**: DINOv2 ViT-S/14 feature matching (2D vision-transformer cosine similarity)
- **Avenue C**: PnP RANSAC peak constellation (already tested locally — loaded from Drive)

**All checkpoints saved to `MyDrive/avenue_checkpoints/`. Re-run skips completed work.**

Required Drive files:
- `MyDrive/skyline_db.parquet`
- `MyDrive/street_view/` (images/, masks/, ground_truth.json, annotations.json)
- `MyDrive/sky_segmentation_unet_model.pth`

GPU: T4 (16GB) recommended. Avenue A+B take ~20 min total.

## 1. Setup

In [ ]:
import os, shutil
from pathlib import Path

REPO = Path('/content/SkylineGeolocation')
BRANCH = 'main'

if (REPO / 'src').exists():
    print('Repo already at', REPO)
else:
    print('Cloning repo...')
    if REPO.is_file() or REPO.is_symlink():
        REPO.unlink()
    elif REPO.is_dir():
        shutil.rmtree(REPO)
    REPO.parent.mkdir(parents=True, exist_ok=True)
    import subprocess
    subprocess.run(
        ['git', 'clone', '--depth', '1', '-b', BRANCH,
         'https://github.com/pxrxp/SkylineGeolocation.git', str(REPO)],
        check=True, capture_output=True, text=True
    )
    print('Cloned. Branch:', BRANCH)
import sys
sys.path.insert(0, str(REPO))

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('Drive mounted at /content/drive')

In [ ]:
!pip install -q timm torchvision fastdtw pyarrow geopy
!apt-get install -qq libgl1-mesa-glx libglib2.0-0 2>/dev/null

# SAM 2 install — skip if already available
try:
    import sam2; print('SAM 2 already installed')
except ImportError:
    print('Installing SAM 2...')
    !pip install -q git+https://github.com/facebookresearch/sam2.git

print('Deps ready')

In [ ]:
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive')
CKPT_DIR = DRIVE / 'avenue_checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
STATE_PATH = CKPT_DIR / 'state.json'

links = [
    (DRIVE / 'skyline_db.parquet',
     REPO / 'notebooks/02_SkylineDatabase/output/skyline_db.parquet'),
    (DRIVE / 'street_view',
     REPO / 'data/street_view'),
    (DRIVE / 'sky_segmentation_unet_model.pth',
     REPO / 'data/sky_segmentation_unet_model.pth'),
]

for src, dst in links:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists() and src.exists():
        os.symlink(src, dst)
        print(f'Linked {src.name}')
    elif dst.exists():
        print(f'{dst.name}: already linked')
    else:
        print(f'MISSING: {src}')

In [ ]:
db_path = REPO / 'notebooks/02_SkylineDatabase/output/skyline_db.parquet'
if db_path.exists():
    size_mb = db_path.stat().st_size / 1024 / 1024
    with open(db_path, 'rb') as f:
        f.seek(0); hdr = f.read(4)
        f.seek(-8, 2); ftr = f.read()
    valid = hdr == b'PAR1' and ftr[:4] == b'PAR1'
    print(f'DB: {size_mb:.0f}MB, valid={valid}')
else:
    print('DB MISSING — cannot proceed')

In [ ]:
import json, time
from pathlib import Path

class CheckpointManager:
    """Per-sample checkpointing to Drive. Crash-safe: state written after each sample."""
    def __init__(self, state_path):
        self.path = Path(state_path)
        self.state = {}
        if self.path.exists():
            try:
                self.state = json.loads(self.path.read_text())
            except (json.JSONDecodeError, OSError):
                self.state = {}
    def is_done(self, avenue, sample_id):
        return self.state.get(avenue, {}).get(sample_id, {}).get('status') == 'ok'
    def mark_done(self, avenue, sample_id, **meta):
        self.state.setdefault(avenue, {})[sample_id] = {
            'status': 'ok', 'time': time.strftime('%Y-%m-%d %H:%M:%S'), **meta
        }
        self.save()
    def save(self):
        self.path.parent.mkdir(parents=True, exist_ok=True)
        tmp = self.path.with_suffix('.tmp')
        tmp.write_text(json.dumps(self.state, indent=2, default=str))
        tmp.rename(self.path)  # atomic on same filesystem
    def summary(self):
        for ave in self.state:
            done = sum(1 for v in self.state[ave].values() if v.get('status') == 'ok')
            total = len(self.state[ave])
            print(f'  {ave}: {done}/{total} samples completed')

ck = CheckpointManager(STATE_PATH)
print('Checkpoint state:')
ck.summary() if ck.state else print('  Fresh start')

## 2. Avenue A: SAM 2 Zero-Shot Sky Segmentation

Replace U-Net 256×256 masks with SAM 2 full-res 1080×720 masks.
Point prompts at top of frame (sky) + bottom (terrain).

In [ ]:
import json, os, sys, time
import numpy as np
from pathlib import Path
from PIL import Image

sys.path.insert(0, str(REPO))
from src.query_profile import extract_elevation_profile
from scripts.fixes_eval import Rx, mask_from_ann, DB_PATH, GT_FILE, ANNOT_FILE, CALIB_FILE

IMAGE_DIR = REPO / 'data/street_view/images'
MASK_DIR_A = CKPT_DIR / 'avenue_a_masks'
PROFILE_DIR_A = CKPT_DIR / 'avenue_a_profiles'
MASK_DIR_A.mkdir(parents=True, exist_ok=True)
PROFILE_DIR_A.mkdir(parents=True, exist_ok=True)

gt = json.load(open(REPO / 'data/street_view/ground_truth.json'))
ann = json.load(open(REPO / 'data/street_view/annotations.json'))['annotations']
calib = json.load(open(REPO / 'data/street_view/calibrated_ground_truth.json'))
sids = [s for s in ann if s in gt and (IMAGE_DIR / f'{s}.png').exists()][:17]
print(f'Samples: {len(sids)}')

need_a = [s for s in sids if not ck.is_done('avenue_a', s)]
print(f'Avenue A remaining: {len(need_a)}/{len(sids)}')

if need_a:
    import torch
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}')

    sam2_model = build_sam2(
        model_cfg='sam2_hiera_l.yaml',
        ckpt='sam2_hiera_large.pt',  # auto-downloaded
        device=str(device)
    )
    predictor = SAM2ImagePredictor(sam2_model)

    for si, sid in enumerate(need_a):
        t0 = time.time()
        g = gt[sid]
        img = np.array(Image.open(IMAGE_DIR / f'{sid}.png').convert('RGB'))
        H, W = img.shape[:2]

        predictor.set_image(img)

        # Point prompts: sky at top center, terrain at bottom center + corners
        pts = np.array([[W//2, 5], [W//2, H-5], [10, H-5], [W-10, H-5]])
        labels = np.array([1, 0, 0, 0])  # 1=sky, 0=terrain

        masks, scores, _ = predictor.predict(
            point_coords=pts, point_labels=labels,
            multimask_output=False
        )
        mask = (masks[0] * 255).astype(np.uint8)  # sky=255, terrain=0

        # Save mask
        mask_path = MASK_DIR_A / f'{sid}.png'
        Image.fromarray(mask).save(mask_path)

        # Extract profile
        tilt = np.array(g['cam_R_tilt'])
        dp = float(calib.get(sid, {}).get('delta_pitch_deg', 0.0))
        R_cal = Rx(dp) @ tilt
        # Invert: SAM2 mask sky=255→need sky=0 for extract_elevation_profile convention
        mask_for_profile = 255 - mask
        pr = extract_elevation_profile(
            mask_for_profile, fov_y_deg=g['fov_y_deg'], r_tilt=R_cal, bin_deg=0.5
        )
        if pr['ok']:
            np.save(PROFILE_DIR_A / f'{sid}.npy', np.array(pr['profile'], dtype=np.float32))

        ck.mark_done('avenue_a', sid,
                     mask_path=str(mask_path),
                     profile_ok=pr['ok'],
                     profile_len=len(pr['profile']) if pr['ok'] else 0,
                     elapsed_s=round(time.time()-t0, 1))
        print(f'  [{si+1}/{len(need_a)}] {sid[:25]:25s} mask={mask.shape} '
              f'profile={"OK" if pr["ok"] else pr["status"]} '
              f'({time.time()-t0:.1f}s)', flush=True)

    del predictor, sam2_model
    torch.cuda.empty_cache()
    print('Avenue A complete')
else:
    print('Avenue A: all samples already checkpointed')

## 3. Avenue B: DINOv2 Feature Matching

Encode query photos + DB horizon silhouette images with DINOv2 ViT-S/14.
Compare cosine similarity to rank candidate VPs.

In [ ]:
import json, os, sys, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from PIL import Image
from geopy.distance import geodesic
import torch
from torchvision import transforms as T

sys.path.insert(0, str(REPO))
from src.horizon_format import decode_horizon_uint8

# Load geometry
meta = pd.read_parquet(DB_PATH, columns=['lon', 'lat'])
vp_lon = meta['lon'].to_numpy()
vp_lat = meta['lat'].to_numpy()
del meta

gt = json.load(open(REPO / 'data/street_view/ground_truth.json'))
sids = [s for s in ann if s in gt and (IMAGE_DIR / f'{s}.png').exists()][:17]
need_b = [s for s in sids if not ck.is_done('avenue_b', s)]
print(f'Avenue B remaining: {len(need_b)}/{len(sids)}')

RESULTS_DIR_B = CKPT_DIR / 'avenue_b_results'
RESULTS_DIR_B.mkdir(parents=True, exist_ok=True)

if need_b:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}')

    dino_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14', pretrained=True)
    dino_model = dino_model.to(device).eval()
    dino_transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    def horizon_to_silhouette(horizon, W=224, H=224):
        L = len(horizon)
        img = np.zeros((H, W), dtype=np.uint8)
        for c in range(W):
            col_az = (c / W) * 360.0
            h_idx = int((col_az % 360.0) / (360.0 / L)) % L
            elev = horizon[h_idx]
            row = int(H * (1.0 - (elev + 10.0) / 100.0))
            row = max(0, min(H - 1, row))
            img[row:, c] = 255
        return img

    def encode_batch(images):
        tensors = []
        for img in images:
            if isinstance(img, np.ndarray):
                img = Image.fromarray(img).convert('RGB')
            tensors.append(dino_transform(img))
        x = torch.stack(tensors).to(device)
        with torch.no_grad():
            feat = dino_model(x)
        return feat.cpu().numpy()

    def cosine_sim(A, B):
        An = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-8)
        Bn = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-8)
        return An @ Bn.T

    def fetch_horizons(idx_list):
        pf = pq.ParquetFile(DB_PATH)
        sizes = [pf.metadata.row_group(i).num_rows for i in range(pf.num_row_groups)]
        starts = np.concatenate([[0], np.cumsum(sizes)])[:-1]
        groups = {}
        for vi in idx_list:
            rg = int(np.searchsorted(starts, vi, side='right') - 1)
            groups.setdefault(rg, []).append(vi)
        out = {}
        for rg, vis in groups.items():
            raw = pf.read_row_group(rg, columns=['raw_horizon_deg']).to_pandas()['raw_horizon_deg'].to_numpy()
            for vi in vis:
                out[vi] = decode_horizon_uint8(raw[vi - starts[rg]])
        return out

    # Preload corr caches
    cache_dir = REPO / 'data/eval/cache'

    for si, sid in enumerate(need_b):
        t0 = time.time()
        g = gt[sid]
        tlat, tlon = g['true_lat'], g['true_lon']
        vp = g['closest_viewpoint_id']

        # Encode query photo
        q_img = Image.open(IMAGE_DIR / f'{sid}.png').convert('RGB')
        q_feat = encode_batch([q_img])

        # Build candidate set from corr cache
        corr_path = cache_dir / f'{sid}_corr.npz'
        if not corr_path.exists():
            print(f'  [{si+1}/{len(need_b)}] {sid}: no corr cache, SKIP', flush=True)
            continue
        corr = np.load(corr_path)['corr']
        top500 = np.argsort(corr)[-500:]
        dlat = vp_lat - tlat
        dlon = vp_lon - tlon
        approx_km = np.sqrt(dlat**2 + dlon**2) * 111.0
        near200 = np.argsort(approx_km)[:200]
        cand = np.unique(np.concatenate([top500, [vp], near200])).astype(int)

        hdict = fetch_horizons(list(cand))
        cand_h = [hdict[int(c)] for c in cand if int(c) in hdict]
        cand_v = np.array([int(c) for c in cand if int(c) in hdict])
        if len(cand_h) == 0:
            continue

        # Encode DB silhouettes in batches
        sil_imgs = [horizon_to_silhouette(h) for h in cand_h]
        db_feats = []
        for bi in range(0, len(sil_imgs), 64):
            db_feats.append(encode_batch(sil_imgs[bi:bi+64]))
        db_feats = np.concatenate(db_feats, axis=0)

        sims = cosine_sim(q_feat, db_feats)[0]
        top5_i = np.argsort(sims)[-5:][::-1]
        top5_vps = cand_v[top5_i]
        top1_err = geodesic((tlat, tlon), (vp_lat[top5_vps[0]], vp_lon[top5_vps[0]])).meters / 1000

        true_i = np.where(cand_v == vp)[0]
        true_rank = int(np.sum(sims > sims[true_i[0]])) + 1 if len(true_i) > 0 else -1
        true_sim = float(sims[true_i[0]]) if len(true_i) > 0 else -1.0

        result = {
            'sid': sid, 'top1_err_km': round(top1_err, 1),
            'true_rank': true_rank, 'sim_true': round(true_sim, 4),
            'top5_vp_ids': [int(v) for v in top5_vps],
            'top5_errors_km': [round(geodesic((tlat, tlon),
                (vp_lat[int(v)], vp_lon[int(v)])).meters / 1000, 1) for v in top5_vps],
            'n_candidates': len(cand_v),
        }
        (RESULTS_DIR_B / f'{sid}.json').write_text(json.dumps(result, indent=2))
        ck.mark_done('avenue_b', sid, top1_err_km=top1_err, true_rank=true_rank)
        print(f'  [{si+1}/{len(need_b)}] {sid[:25]:25s} top1={top1_err:6.1f}km '
              f'rank={true_rank:5d} sim={true_sim:.4f} '
              f'({time.time()-t0:.1f}s)', flush=True)

    del dino_model
    torch.cuda.empty_cache()
    print('Avenue B complete')
else:
    print('Avenue B: all samples already checkpointed')

## 4. Aggregate Results

In [ ]:
import json, os
import numpy as np
from pathlib import Path
from geopy.distance import geodesic
import sys

sys.path.insert(0, str(REPO))

print('=== Checkpoint Status ===')
ck.summary()

# Avenue A summary
print('\n=== Avenue A: SAM 2 Masks ===')
a_results = []
for f in sorted((PROFILE_DIR_A).glob('*.npy')):
    sid = f.stem
    prof = np.load(f)
    a_results.append({'sid': sid, 'profile_len': len(prof), 'profile_std': round(float(np.std(prof)), 2)})
if a_results:
    print(f'  {len(a_results)} profiles extracted')
    stds = [r['profile_std'] for r in a_results]
    print(f'  Profile std: med={np.median(stds):.2f}  min={np.min(stds):.2f}  max={np.max(stds):.2f}')
else:
    print('  No profiles yet')

# Avenue B summary
print('\n=== Avenue B: DINOv2 Feature Matching ===')
b_results = []
for f in sorted(RESULTS_DIR_B.glob('*.json')):
    b_results.append(json.loads(f.read_text()))
if b_results:
    errs = np.array([r['top1_err_km'] for r in b_results])
    ranks = np.array([r['true_rank'] for r in b_results if r['true_rank'] > 0])
    print(f'  Samples: {len(b_results)}')
    print(f'  Top-1 error: med={np.median(errs):.1f}km  '
          f'<1km={int(np.sum(errs < 1))}/{len(errs)}  '
          f'<5km={int(np.sum(errs < 5))}/{len(errs)}  '
          f'<10km={int(np.sum(errs < 10))}/{len(errs)}')
    if len(ranks) > 0:
        print(f'  True-VP rank: med={int(np.median(ranks)):d}  mean={np.mean(ranks):.0f}')
    print('\n  Per-sample:')
    for r in sorted(b_results, key=lambda x: x['top1_err_km']):
        print(f'    {r["sid"][:25]:25s} err={r["top1_err_km"]:6.1f}km  rank={r["true_rank"]:5d}')
else:
    print('  No results yet')

# Save combined summary
summary = {
    'avenue_a': a_results,
    'avenue_b': b_results,
}
(CKPT_DIR / 'combined_summary.json').write_text(json.dumps(summary, indent=2))
print(f'\nSummary saved to {CKPT_DIR / "combined_summary.json"}')

## Notes

- **Checkpointing**: After each sample, state is written to `MyDrive/avenue_checkpoints/state.json`. If Colab crashes, re-run all cells — completed samples are skipped.
- **GPU Memory**: Avenue A releases SAM 2 before Avenue B loads DINOv2. T4 16GB is sufficient.
- **Account switching**: All results live on Drive. If GPU hours exhaust, switch account, re-mount Drive, re-run — picks up from checkpoint.
- **Avenue C (PnP)**: Already tested locally (10/18 success, 7.5km med). Results in AGENTS.md.